# 🔍 SHAP Integration Demo - TimeLense

## Model Explainability and Feature Importance Analysis

This notebook demonstrates how to use SHAP (SHapley Additive exPlanations) with TimeLense for model interpretability.

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# SHAP and ML libraries
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

print("✅ All libraries imported successfully")
print(f"SHAP version: {shap.__version__}")

In [ ]:
# Load processed data from TimeLense
try:
    df = pd.read_csv('../data/processed/events_cleaned.csv', parse_dates=['event_time'])
    print(f"✅ Loaded {len(df):,} events from processed data")
    print(f"Time range: {df['event_time'].min()} to {df['event_time'].max()}")
    
    # Show sample
    display(df.head())
    
except FileNotFoundError:
    print("❌ Processed data not found. Please run the data preparation notebook first.")
    df = None

In [ ]:
if df is not None:
    # Create daily aggregation for prediction task
    daily_data = df.groupby(df['event_time'].dt.date).agg({
        'user_id': 'nunique',
        'event_type': 'count',
        'price': ['sum', 'mean', 'std'],
        'product_id': 'nunique',
        'user_session': 'nunique'
    }).reset_index()
    
    # Flatten column names
    daily_data.columns = ['date', 'unique_users', 'total_events', 'total_revenue', 'avg_price', 'price_std', 'unique_products', 'unique_sessions']
    daily_data['date'] = pd.to_datetime(daily_data['date'])
    
    print(f"✅ Created daily aggregation: {len(daily_data)} days")
    display(daily_data.head())
    
    # Visualize the target variable
    fig = px.line(daily_data, x='date', y='total_revenue', 
                  title='Daily Revenue Over Time',
                  labels={'total_revenue': 'Total Revenue ($)'})
    fig.show()

In [ ]:
# Initialize SHAP explainer
from intelligence.shap_explainer import TimeLenseSHAPExplainer

# Create explainer instance
explainer = TimeLenseSHAPExplainer(daily_data, 'date', 'total_revenue')

print("🔧 Preparing features with temporal components...")
processed_data = explainer.prepare_features(include_temporal=True, include_lags=True)

print(f"✅ Prepared {len(explainer.feature_names)} features for analysis")
print(f"Features: {explainer.feature_names}")

# Show processed data sample
display(processed_data.head())

In [ ]:
# Train multiple models for comparison
print("🤖 Training multiple models...")
model_results = explainer.train_models(test_size=0.3)

# Display model performance
performance_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'R² Score': [r['r2'] for r in model_results.values()],
    'MSE': [r['mse'] for r in model_results.values()]
}).sort_values('R² Score', ascending=False)

print("🏆 Model Performance Comparison:")
display(performance_df)

# Select best model
best_model = performance_df.iloc[0]['Model']
print(f"\n🥇 Best performing model: {best_model} (R² = {performance_df.iloc[0]['R² Score']:.3f})")

In [ ]:
# Create SHAP explainers
print("🔍 Creating SHAP explainers for model interpretability...")
shap_explainers = explainer.create_shap_explainers(model_results)

print(f"✅ Created SHAP explainers for {len(shap_explainers)} models")
print(f"Available models: {list(shap_explainers.keys())}")

In [ ]:
# Generate SHAP summary plot (feature importance)
print(f"📊 Generating SHAP feature importance plot for {best_model}...")

fig_summary = explainer.generate_shap_summary_plot(best_model)
if fig_summary:
    fig_summary.show()
else:
    print("❌ Could not generate summary plot")

In [ ]:
# Generate SHAP waterfall plot for individual predictions
print("🔍 Analyzing individual predictions with SHAP waterfall plots...")

# Show waterfall for first few instances
for i in range(min(3, len(shap_explainers[best_model]['shap_values']))):
    print(f"\nInstance {i}:")
    fig_waterfall = explainer.generate_shap_waterfall_plot(best_model, i)
    if fig_waterfall:
        fig_waterfall.show()
    else:
        print(f"❌ Could not generate waterfall plot for instance {i}")

In [ ]:
# Generate SHAP dependence plots for key features
print("📈 Analyzing feature effects with SHAP dependence plots...")

# Get top 3 most important features
shap_values = shap_explainers[best_model]['shap_values']
feature_importance = np.abs(shap_values).mean(axis=0)
top_features_idx = np.argsort(feature_importance)[-3:][::-1]
top_features = [explainer.feature_names[i] for i in top_features_idx]

print(f"Top 3 most important features: {top_features}")

# Create dependence plots
for feature in top_features:
    print(f"\nFeature: {feature}")
    fig_dep = explainer.generate_shap_dependence_plot(best_model, feature)
    if fig_dep:
        fig_dep.show()
    else:
        print(f"❌ Could not generate dependence plot for {feature}")

In [ ]:
# Generate textual insights from SHAP analysis
print("💡 Extracting actionable insights from SHAP analysis...")

insights = explainer.get_shap_insights(best_model)

print(f"\n📋 Generated {len(insights)} key insights:")
print("=" * 50)

for i, insight in enumerate(insights, 1):
    print(f"\n{i}. {insight['title']}")
    print(f"   {insight['text']}")
    print(f"   Importance: {insight['importance']}/100")
    print(f"   Category: {insight['category']}")

In [ ]:
# Summary and recommendations
print("🎉 SHAP Analysis Complete!")
print("=" * 40)
print(f"\n📊 Analysis Summary:")
print(f"• Dataset: {len(daily_data)} days of e-commerce data")
print(f"• Target variable: {explainer.target_col}")
print(f"• Features analyzed: {len(explainer.feature_names)}")
print(f"• Best model: {best_model} (R² = {performance_df.iloc[0]['R² Score']:.3f})")
print(f"• Key insights generated: {len(insights)}")

print(f"\n🚀 Next Steps:")
print("• Use insights to optimize business processes")
print("• Focus on top-contributing features for improvements")
print("• Monitor model explanations over time")
print("• Apply SHAP analysis to other metrics and time periods")

print(f"\n✅ Your predictive models are now fully explainable with SHAP!")